# **ViT로 이미지 자동 Tagging**



---



## **이미지 데이터 준비**

### **1) 이미지 데이터 준비하기**

### ✅ **이미지 데이터 다운로드**
1.  https://www.pexels.com/api/ 회원가입
2. "Get Started" 클릭 후 API Key 복사
3. **SEARCH_QUERY** : ex) cat(jpg), dog(jpg), car(gif), flower(png)

In [ ]:
from tqdm import tqdm  # tqdm: 긴 시간 동안 실행되는 반복문의 진행 상황을 사용자에게 알려주는 라이브러리
import time

for i in tqdm(range(100)):
    time.sleep(0.1) # 각 반복에 약간의 시간 지연을 줍니다.
print("작업 완료!")

In [ ]:
import os
import requests
from tqdm import tqdm

# ➡️ Pexels API Key 입력
PEXELS_API_KEY = 'Czm6SeJLlWd5ZIm3wbQZBLTZmhBT4MQl1vtUzxzgxxEtkT2MBDsMGRq7'  # 여기에 본인의 키를 입력하세요

# ➡️ 검색 키워드 및 저장 폴더
SEARCH_QUERY = 'city'   # 검색할 이미지 키워드
SAVE_FOLDER = 'images'
IMAGES_TO_DOWNLOAD = 100  # 총 몇 장 다운로드할 것인지

# ➡️ API 요청 함수
def search_images(query, per_page=30, page=1):
    headers = {
        "Authorization": PEXELS_API_KEY
    }
    url = f"https://api.pexels.com/v1/search?query={query}&per_page={per_page}&page={page}"
    response = requests.get(url, headers=headers)
    return response.json()

# ➡️ 이미지 다운로드 함수
def download_image(url, save_path):
    r = requests.get(url, stream=True)
    if r.status_code == 200:
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(1024):  # 1024바이트 크기의 덩어리(chunk)로 나누어 순회 저장
                f.write(chunk)

# ➡️ 이미지 검색 & 다운로드 실행
os.makedirs(SAVE_FOLDER, exist_ok=True)

downloaded = 0
page = 1
while downloaded < IMAGES_TO_DOWNLOAD:
    data = search_images(SEARCH_QUERY, per_page=30, page=page)
    photos = data.get('photos', [])

    if not photos:
        print("이미지를 더 이상 찾을 수 없습니다.")
        break

    for photo in photos:
        image_url = photo['src']['large']
        filename = f"{SEARCH_QUERY}_{downloaded+1}.png"
        save_path = os.path.join(SAVE_FOLDER, filename)
        download_image(image_url, save_path)
        downloaded += 1
        tqdm.write(f"다운로드 완료: {filename}")

        if downloaded >= IMAGES_TO_DOWNLOAD:
            break
    page += 1

print(f"\n총 {downloaded}장의 이미지를 '{SAVE_FOLDER}' 폴더에 저장했습니다.")

### **2) 이미지 데이터 속성**

- 이미지 파일 : JPG, PNG 등 확장자를 갖는 파일
- **이미지 속성**:
    - 파일명(파일명/날짜/이름)
    - 파일 용량 (bytes)
    - 해상도 (Width x Height)
    - 이미지 모드 (RGB, L 등) : '1' (Binary image), 'L' (Grayscale image), 'RGB' (Red, Green, Blue)
    - 밝기 (Grayscale 평균값)
    - 색상 (평균 RGB, 색상 히스토그램 등)

### ✅ **이미지 데이터 속성 추출하기**

In [ ]:
# !pip install pillow numpy pandas

In [ ]:
import os
from PIL import Image
import numpy as np
import pandas as pd

def extract_image_features(image_path):
    try:
        img = Image.open(image_path)
        img_array = np.array(img)

        # ➡️ 기본 속성
        filename = os.path.basename(image_path)
        file_size = os.path.getsize(image_path)  # bytes
        width, height = img.size
        mode = img.mode

        # ➡️ 밝기 (Grayscale 변환 후 평균)
        grayscale = img.convert('L')
        brightness = np.array(grayscale).mean()

        # ➡️ RGB 평균
        if img.mode in ['RGB', 'RGBA']:
            rgb_array = np.array(img.convert('RGB'))
            avg_color = rgb_array.mean(axis=(0, 1))  # R, G, B 평균값
        else:
            avg_color = [None, None, None]

        return {
            '파일명': filename,
            '파일크기(Bytes)': file_size,
            '해상도': f"{width}x{height}",
            '모드': mode,
            '평균밝기': round(brightness, 2),
            'R평균': round(avg_color[0], 2) if avg_color[0] else None,
            'G평균': round(avg_color[1], 2) if avg_color[1] else None,
            'B평균': round(avg_color[2], 2) if avg_color[2] else None
        }

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

# 이미지 폴더 경로
image_folder = 'images'  # 여기에 실제 경로 입력
image_files = [os.path.join(image_folder, f)
               for f in os.listdir(image_folder)
               if f.lower().endswith(('.jpg', '.jpeg', '.png','gif'))]

# 속성 추출
image_data = [extract_image_features(img_path) for img_path in image_files]
image_data = [d for d in image_data if d is not None]

# 데이터프레임 보기 좋게 출력
df = pd.DataFrame(image_data)
display(df)

### ✅ **이미지 시각화**

In [ ]:
# 밝기 기반 정렬
from PIL import Image
import os
import numpy as np


# ➡️ 이미지를 밝기 값으로 정렬하기(평균값)
def get_avg_brightness(image_path):
    image = Image.open(image_path).convert('L')  # L = grayscale
    arr = np.array(image)
    return arr.mean()

image_folder = 'images'     # 이미지 폴더
# image_files = [f for f in os.listdir(image_folder) if f.endswith('.jpg')]
image_files = [f for f in os.listdir(image_folder)]

# 내장 정렬 함수 사용하여 정렬
sorted_images = sorted(
    image_files,
    key=lambda x: get_avg_brightness(os.path.join(image_folder, x)),
    reverse=True
)

print(sorted_images[:3])    # 밝기값 내림차순
print(sorted_images[::10])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# ➡️ 이미지를 나란히 그래프로 시각화하기
def visualize_resized_images(image_paths):
    """
    1.주어진 이미지 리스트의 이미지들을 동일한 크기로 조정(200x200)
    2.2행 3열 (또는 이미지 개수에 따른 열) 그래프에 시각화
    3.1행에는 원본 이미지, 2행에는 grayscale 이미지 나타냄
    """
    target_size = (200, 200)  # 목표로 하는 이미지 크기 (원하는 가로, 세로 크기로 변경 가능)
    original_images = []
    grayscale_images = []

    for path in image_paths:
        try:
            img = Image.open(path)
            resized_img = img.resize(target_size)       # 크기 변환
            original_images.append(resized_img)
            grayscale_img = resized_img.convert("L")    # Grayscale로 변환
            grayscale_images.append(grayscale_img)
        except FileNotFoundError:
            print(f"경로 '{path}'에 파일이 없습니다.")
            continue


    if len(original_images) > 0:
        num_images = len(original_images)
        fig, axes = plt.subplots(2, num_images, figsize=(2 * num_images, 4)) # 2행으로 변경, 전체 그래프 높이 조절

        if num_images == 1:
            axes = [[axes[0]], [axes[1]]] # 이미지가 하나일 경우 axes를 2D 리스트 형태로 처리

        for i in range(num_images):
            # 원본 이미지 시각화 (1행)
            original_img_array = np.array(original_images[i])
            axes[0][i].imshow(original_img_array)
            axes[0][i].set_title(f"Original {i+1}")
            axes[0][i].axis('off')

            # grayscale 이미지 시각화 (2행)
            grayscale_img_array = np.array(grayscale_images[i])
            grayscale_img_array.mean()
            axes[1][i].imshow(grayscale_img_array, cmap='gray') # grayscale 이미지는 cmap='gray' 설정
            axes[1][i].set_title(f"Grayscale {grayscale_img_array.mean():.1f}")  # grayscale 평균값
            axes[1][i].axis('off')

        plt.tight_layout()
        plt.show()
    else:
        print("이미지 리스트가 비어 있습니다.")


# 사용할 이미지 파일 경로 리스트
# image_list = ['cars_7.gif', 'flowers_75.png', 'cars_70.gif'...]  # 이름 직접 지정
image_list = sorted_images[::10][:5]    # 10개씩 건너띄어 5개 이미지
# image_list = [image_folder + '/' + filename for filename in image_list]
image_list = [os.path.join(image_folder, filename) for filename in image_list]

# 함수 호출
visualize_resized_images(image_list)

- **색상(RGB) 평균값 기반 정렬**
<br> 파란색 평균값 기반 정렬

In [ ]:
def get_avg_color(image_path):
    image = Image.open(image_path).resize((50, 50))  # 계산 속도 개선
    arr = np.array(image)
    return arr[:, :, :3].mean(axis=(0, 1))  # RGB 평균  arr[:, :, :3]=[행.열,면]

# # R값 기준 정렬
# sorted_images_by_c = sorted(image_files, key=lambda x: get_avg_color(os.path.join(image_folder, x))[0],reverse=True)
# # G값 기준 정렬
# sorted_images_by_c = sorted(image_files, key=lambda x: get_avg_color(os.path.join(image_folder, x))[1],reverse=True)
# # B값 기준 정렬
sorted_images_by_c = sorted(image_files, key=lambda x: get_avg_color(os.path.join(image_folder, x))[2],reverse=True)

print(sorted_images_by_c[:3])
# print(sorted_images[::10])

# 사용할 이미지 파일 경로 리스트
image_list = sorted_images_by_c[::10][:5]    # 10개씩 건너띄어 5개 이미지
image_list = [image_folder + '/' + filename for filename in image_list]
# image_list = [os.path.join(image_folder, filename) for filename in image_list]

# 함수 호출
visualize_resized_images(image_list)



---



## **AI로 이미지 태깅하기**
- [주의!] 이미지 데이터가 이미 해당 폴더(images)에 있는 상태에 진행한다.
- 1) --> 2) --> 3) 순차적으로 실행한다.



---



### ✅ **1) 이미지 정보 추출하기**
- AI 이미지 분류 태그명 (예: "Golden Retriever", "Mountain Bike")
- Feature Vector (ViT 모델의 출력 벡터)
- **모델명** : **google/vit-base-patch16-224**
    - Vision Transformer (ViT) 계열의 대표적인 이미지 분류(Image Classification) 모델
    - Google Research가 제안했으며, Transformer를 자연어 처리(NLP)뿐만 아니라 이미지 인식(CV) 에도 성공적으로 적용한 첫 모델 중 하나
    - 입력 이미지 크기 : 	224 × 224 픽셀
    - 패치 크기 (Patch Size):	16 × 16 픽셀
    - 사전 학습 데이터 : ImageNet-21k (21,000개 클래스, 약 1,400만 이미지)
    - Fine-tuning 대상 : ImageNet-1k (1,000개 클래스, 100만 이미지)
- 작업 순서
    1. **이미지 분할** : 입력 이미지(224×224)를 16×16 크기의 작은 패치(총 14×14=196개)로 나눔
    2. **패치 임베딩**(Patch Embedding) : 각 패치를 1D 벡터로 변환하여 “토큰”처럼 사용
    3. **위치 임베딩**(Positional Embedding) : 패치의 위치 정보를 더하여 순서 개념을 보존
    4. **Transformer Encoder** : NLP의 Transformer와 동일한 구조로, 다중 어텐션(Multi-head Attention)을 통해 전체 이미지의 문맥적 관계 학습
    5. **분류 토큰**(Class Token) : 모든 패치 토큰의 정보를 요약하여 최종적으로 이미지의 클래스를 예측

In [ ]:
# 1. 라이브러리 설치 (필요 시)
#!pip install transformers torch pandas pillow --quiet

# 2. 라이브러리 import
import os
import torch
import pandas as pd
from PIL import Image
from transformers import ViTImageProcessor, ViTModel, ViTForImageClassification

# 3. 모델 로드
model_name = "google/vit-base-patch16-224"
feature_extractor = ViTImageProcessor.from_pretrained(model_name)
model_cls = ViTForImageClassification.from_pretrained(model_name)
model_vec = ViTModel.from_pretrained(model_name)

model_cls.eval()
model_vec.eval()

# 4. 이미지 디렉터리 설정
image_folder = "images"
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# 5. 특징 추출 함수
def extract_features(img_path):
    image = Image.open(img_path).convert("RGB")
    inputs = feature_extractor(images=image, return_tensors="pt")

    # 분류 태그
    with torch.no_grad():
        outputs_cls = model_cls(**inputs)
        logits = outputs_cls.logits
        pred_label_id = logits.argmax(-1).item()
        label_name = model_cls.config.id2label[pred_label_id]

    # feature vector 추출
    with torch.no_grad():
        outputs_vec = model_vec(**inputs)
        cls_vector = outputs_vec.last_hidden_state[:, 0, :].squeeze().numpy()  # [CLS] token

    return label_name, cls_vector

# 6. 모든 이미지 처리
data = []

for fname in image_files:
    path = os.path.join(image_folder, fname)
    label, vec = extract_features(path)
    row = {
        "filename": fname,
        "label": label,
        **{f"feat_{i}": v for i, v in enumerate(vec)}  # feature vector 확장
    }
    data.append(row)

# 7. pandas로 정리
df = pd.DataFrame(data)

# 8. 결과 미리보기
print(df[['filename', 'label']].head())

# 9. CSV로 저장
df.to_csv("image_features_with_labels.csv", index=False)

### ✅ **2) 분류기법으로 자동 태깅** 및 정렬

In [ ]:
# 1. 필요한 라이브러리 설치
# !pip install transformers torch torchvision matplotlib pillow --quiet


# 2. 라이브러리 import
import os
import torch
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification
import matplotlib.pyplot as plt

# 3. 모델과 전처리기 로드 (사전학습된 이미지 분류 모델 사용)
img_path = "images"
model_name = "google/vit-base-patch16-224"
feature_extractor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(model_name)


# 4. 분류 함수 정의
def classify_image(img_path):
    image = Image.open(img_path).convert("RGB")
    inputs = feature_extractor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    predicted_label = logits.argmax(-1).item()
    label_name = model.config.id2label[predicted_label]
    return label_name

# 5. 이미지 디렉터리 지정 및 태깅
image_folder = "images"  # 예: 로컬 디렉터리 또는 colab에 업로드한 폴더
tagged_images = []

for file_name in sorted(os.listdir(image_folder)):
    if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(image_folder, file_name)
        tag = classify_image(img_path)
        tagged_images.append((file_name, tag))

# 6. 태그 기준으로 정렬
tagged_images.sort(key=lambda x: x[1])  # 알파벳 순 정렬

# 7. 결과 시각화
def display_images_by_tag(tagged_images, image_folder, cols=4):
    rows = (len(tagged_images) + cols - 1) // cols
    plt.figure(figsize=(15, rows * 3))
    for i, (fname, tag) in enumerate(tagged_images):
        img = Image.open(os.path.join(image_folder, fname))
        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.title(f"{tag}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# 8. 출력
display_images_by_tag(tagged_images, image_folder)



---



### ✅ **3) 특징 벡터를 추출하여 클러스터링**
- n_clusters = 6  # 원하는 클러스터 수

In [ ]:
# 1. 라이브러리 설치
# !pip install transformers torch torchvision scikit-learn matplotlib pillow --quiet


# 2. 라이브러리 import
import os
import torch
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from transformers import ViTImageProcessor, ViTModel
import numpy as np


# 3. 모델 및 전처리기 로드 (분류용이 아닌 feature vector용)
model_name = "google/vit-base-patch16-224"
feature_extractor = ViTImageProcessor.from_pretrained(model_name)
model = ViTModel.from_pretrained(model_name)
model.eval()  # 추론 모드


# 4. 특징 벡터 추출 함수
def extract_feature_vector(img_path):
    image = Image.open(img_path).convert("RGB")
    inputs = feature_extractor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # [CLS] token의 출력 벡터 사용 (1, hidden_dim)
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    return cls_embedding


# 5. 이미지 디렉토리에서 특징 추출
image_folder = "images"  # 본인의 이미지 디렉터리 지정
image_vectors = []
image_files = []

for file_name in sorted(os.listdir(image_folder)):
    if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        path = os.path.join(image_folder, file_name)
        vec = extract_feature_vector(path)
        image_vectors.append(vec)
        image_files.append(file_name)


# 6. KMeans 클러스터링 수행
n_clusters = 6  # 원하는 클러스터 수
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(image_vectors)


# 7. 클러스터링 결과 정리
clustered_images = list(zip(image_files, clusters))
clustered_images.sort(key=lambda x: x[1])  # 클러스터 번호 기준 정렬


# 8. 시각화 함수
def display_clustered_images(clustered_images, image_folder, cols=4):
    rows = (len(clustered_images) + cols - 1) // cols
    plt.figure(figsize=(15, rows * 3))
    for i, (fname, cluster_id) in enumerate(clustered_images):
        img = Image.open(os.path.join(image_folder, fname))
        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.title(f"Cluster {cluster_id}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


# 9. 결과 출력
display_clustered_images(clustered_images, image_folder)